In [4]:
"""
Amazon E-Commerce Sales Data - Complete Preprocessing Pipeline
==============================================================
Run this script directly with: python amazon_sales_preprocessing.py

If the dataset is not found locally, it will prompt you to download it.
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
RAW_DATA_PATH   = "data/Amazon Sale Report.csv"
CLEAN_DATA_PATH = "data/cleaned_amazon_sales.csv"
OUTPUT_PATH     = "data/preprocessed_amazon_sales.csv"

os.makedirs("data",    exist_ok=True)
os.makedirs("outputs", exist_ok=True)


# ─────────────────────────────────────────────
# 0. AUTO-GENERATE SAMPLE DATA (fallback)
# ─────────────────────────────────────────────
def generate_sample_data(n=5000):
    """
    Creates realistic synthetic Amazon sales data so the pipeline
    can run even without the real CSV file.
    """
    print("\n  Generating synthetic sample data …")
    np.random.seed(42)
    rng = pd.date_range("2022-04-01", "2022-06-30", freq="h")

    categories  = ["Set", "kurta", "Western Dress", "Top", "Ethnic Dress",
                   "Blouse", "Bottom", "Saree", "Dupatta", "Sock"]
    sizes       = ["XS", "S", "M", "L", "XL", "XXL", "3XL", "4XL", "5XL", "6XL", "Free"]
    statuses    = ["Shipped", "Cancelled", "Shipped - Delivered to Buyer",
                   "Shipped - Returned to Seller", "Pending"]
    fulfilments = ["Amazon", "Merchant"]
    channels    = ["Amazon.in", "Non-Amazon"]
    states      = ["Maharashtra", "Karnataka", "Telangana", "Tamil Nadu",
                   "Uttar Pradesh", "Delhi", "Gujarat", "Rajasthan"]
    cities      = ["Mumbai", "Bengaluru", "Hyderabad", "Chennai",
                   "Lucknow", "New Delhi", "Ahmedabad", "Jaipur"]

    df = pd.DataFrame({
        "Order ID"       : [f"405-{np.random.randint(1e6,9e6):.0f}-{np.random.randint(1e6,9e6):.0f}" for _ in range(n)],
        "Date"           : np.random.choice(rng, n),
        "Status"         : np.random.choice(statuses,    n, p=[0.55, 0.15, 0.20, 0.05, 0.05]),
        "Fulfilment"     : np.random.choice(fulfilments, n, p=[0.70, 0.30]),
        "Sales Channel"  : np.random.choice(channels,    n, p=[0.90, 0.10]),
        "Category"       : np.random.choice(categories,  n),
        "Size"           : np.random.choice(sizes,        n),
        "Qty"            : np.random.choice([1, 2, 3, 4, 5], n, p=[0.60, 0.20, 0.10, 0.05, 0.05]),
        "Amount"         : np.round(np.random.lognormal(6.2, 0.7, n), 2),
        "ship-city"      : np.random.choice(cities,  n),
        "ship-state"     : np.random.choice(states,  n),
        "ship-postal-code": np.random.randint(100000, 999999, n),
        "B2B"            : np.random.choice([True, False], n, p=[0.05, 0.95]),
        "promotion-ids"  : np.random.choice(["No Promotion", "IN_Core_1", "IN_Summer_2", ""], n,
                                             p=[0.60, 0.15, 0.15, 0.10]),
        "ASIN"           : ["B0" + "".join(np.random.choice(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"), 8))
                            for _ in range(n)],
        "Style"          : [f"JNE{np.random.randint(1000,9999)}" for _ in range(n)],
    })

    # Inject ~3 % missing values in Amount / Qty
    for col in ["Amount", "Qty"]:
        idx = np.random.choice(df.index, size=int(n * 0.03), replace=False)
        df.loc[idx, col] = np.nan

    df.to_csv(RAW_DATA_PATH, index=False)
    print(f"  Sample data saved to: {RAW_DATA_PATH}  ({n:,} rows)")
    return df


# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
def load_data():
    print("\n" + "="*60)
    print(" LOADING DATA")
    print("="*60)

    # Try common file names
    candidates = [
        RAW_DATA_PATH,
        "data/Amazon Sales Dataset.csv",
        "data/amazon_sales.csv",
        "Amazon Sale Report.csv",
        "Amazon Sales Dataset.csv",
        CLEAN_DATA_PATH,
    ]

    for path in candidates:
        if os.path.exists(path):
            df = pd.read_csv(path, low_memory=False)
            print(f"  Loaded  : {path}")
            print(f"  Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
            return df

    # Nothing found → generate synthetic data
    print("  No CSV found. Using synthetic sample data instead.")
    print("  (Place your real CSV at: data/Amazon Sale Report.csv to use real data)\n")
    return generate_sample_data()


# ─────────────────────────────────────────────
# 2. EXPLORE DATA
# ─────────────────────────────────────────────
def explore_data(df):
    print("\n" + "="*60)
    print(" DATA OVERVIEW")
    print("="*60)
    print(f"\n[Column Names]\n{list(df.columns)}")
    print(f"\n[Data Types]\n{df.dtypes.to_string()}")
    print(f"\n[First 3 rows]\n{df.head(3).to_string()}")

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n[Missing Values]\n{missing if len(missing) else 'None'}")
    print(f"\n[Duplicated rows]: {df.duplicated().sum():,}")
    return df


# ─────────────────────────────────────────────
# 3. CLEAN DATA
# ─────────────────────────────────────────────
def clean_data(df):
    print("\n" + "="*60)
    print(" DATA CLEANING")
    print("="*60)
    rows_before = len(df)

    # 3a. Standardise column names
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[\s\-]+', '_', regex=True)
    )
    print(f"  Columns renamed: {list(df.columns)}")

    # 3b. Parse date column
    date_cols = [c for c in df.columns if 'date' in c]
    if date_cols:
        dc = date_cols[0]
        df[dc] = pd.to_datetime(df[dc], errors='coerce')
        print(f"  Parsed '{dc}' as datetime")

    # 3c. Drop duplicates
    df = df.drop_duplicates()
    print(f"  Dropped duplicates: {rows_before - len(df):,} rows removed")

    # 3d. Force numeric on amount / qty
    for col in ['amount', 'qty']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3e. Fill missing numerics with median
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        n_missing = df[col].isnull().sum()
        if n_missing:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            print(f"  Filled '{col}' ({n_missing:,} NaN) → median = {med:.2f}")

    # 3f. Fill missing categoricals with 'Unknown'
    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        n_missing = df[col].isnull().sum()
        if n_missing:
            df[col] = df[col].fillna('Unknown')
            print(f"  Filled '{col}' ({n_missing:,} NaN) → 'Unknown'")

    # 3g. Remove non-positive amounts
    if 'amount' in df.columns:
        bad = (df['amount'] <= 0) | df['amount'].isnull()
        print(f"  Removed {bad.sum():,} rows with amount ≤ 0 or NaN")
        df = df[~bad]

    # 3h. Remove zero / negative qty
    if 'qty' in df.columns:
        bad = (df['qty'] <= 0) | df['qty'].isnull()
        print(f"  Removed {bad.sum():,} rows with qty ≤ 0 or NaN")
        df = df[~bad]

    # 3i. Save cleaned file
    df.to_csv(CLEAN_DATA_PATH, index=False)
    print(f"\n  Cleaned data saved → {CLEAN_DATA_PATH}")
    print(f"  Shape after cleaning: {df.shape[0]:,} × {df.shape[1]}")
    return df


# ─────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────
def engineer_features(df):
    print("\n" + "="*60)
    print(" FEATURE ENGINEERING")
    print("="*60)

    date_cols = [c for c in df.columns if 'date' in c]

    # 4a. Revenue
    if 'amount' in df.columns and 'qty' in df.columns:
        df['revenue'] = df['amount'] * df['qty']
        print("  Created 'revenue' (amount × qty)")

    # 4b. Profit proxy (30 % margin)
    if 'revenue' in df.columns:
        df['profit'] = df['revenue'] * 0.30
        print("  Created 'profit' (revenue × 0.30)")

    # 4c. Date features
    if date_cols:
        dc = date_cols[0]
        df['day_of_week']  = df[dc].dt.dayofweek
        df['day_of_month'] = df[dc].dt.day
        df['month']        = df[dc].dt.month
        df['quarter']      = df[dc].dt.quarter
        df['year']         = df[dc].dt.year
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        print("  Created date features: day_of_week, day_of_month, month, quarter, year, is_weekend")

    # 4d. Price bucket
    if 'amount' in df.columns:
        df['price_bucket'] = pd.cut(
            df['amount'],
            bins=[0, 200, 500, 1000, float('inf')],
            labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
        )
        print("  Created 'price_bucket'")

    # 4e. Promotion flag
    promo_cols = [c for c in df.columns if 'promotion' in c]
    if promo_cols:
        df['has_promotion'] = (
            ~df[promo_cols[0]].isin(['No Promotion', 'Unknown', ''])
        ).astype(int)
        print("  Created 'has_promotion' binary flag")

    # 4f. Fulfilled by Amazon flag
    ful_cols = [c for c in df.columns if 'fulfilled' in c or 'fulfilment' in c]
    if ful_cols:
        df['fulfilled_by_amazon'] = (
            df[ful_cols[0]].str.lower() == 'amazon'
        ).astype(int)
        print("  Created 'fulfilled_by_amazon' binary flag")

    # 4g. B2B flag
    b2b_cols = [c for c in df.columns if 'b2b' in c]
    if b2b_cols:
        df['b2b_flag'] = (
            df[b2b_cols[0]].astype(str).str.lower()
              .map({'true': 1, 'false': 0, '1': 1, '0': 0})
              .fillna(0).astype(int)
        )
        print("  Created 'b2b_flag' binary flag")

    print(f"\n  Shape after feature engineering: {df.shape[0]:,} × {df.shape[1]}")
    return df


# ─────────────────────────────────────────────
# 5. ENCODE CATEGORICALS & SAVE
# ─────────────────────────────────────────────
def encode_and_save(df, output_path):
    print("\n" + "="*60)
    print(" ENCODING & SAVING")
    print("="*60)

    from sklearn.preprocessing import LabelEncoder

    # Columns to skip encoding (high-cardinality IDs / free-text)
    skip = {'order_id', 'asin', 'sku', 'style', 'ship_city',
            'ship_postal_code', 'promotion_ids', 'promotion_ids'}

    cat_cols = [
        c for c in df.select_dtypes(include=['object', 'category']).columns
        if c not in skip
    ]

    le = LabelEncoder()
    for col in cat_cols:
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        print(f"  Label-encoded '{col}' → '{col}_enc'")

    df.to_csv(output_path, index=False)
    print(f"\n  Preprocessed data saved → {output_path}")
    print(f"  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == '__main__':
    df = load_data()
    df = explore_data(df)
    df = clean_data(df)
    df = engineer_features(df)
    df = encode_and_save(df, OUTPUT_PATH)

    print("\n" + "="*60)
    print(" PREPROCESSING COMPLETE")
    print("="*60)
    print(f"  Output file : {OUTPUT_PATH}")
    print(f"  Rows        : {df.shape[0]:,}")
    print(f"  Columns     : {df.shape[1]}")
    print("  Key features: revenue, profit, date features,")
    print("                price_bucket, has_promotion,")
    print("                fulfilled_by_amazon, b2b_flag")
    print("="*60)


 LOADING DATA
  Loaded  : data/Amazon Sale Report.csv
  Shape   : 5,000 rows × 16 columns

 DATA OVERVIEW

[Column Names]
['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel', 'Category', 'Size', 'Qty', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'B2B', 'promotion-ids', 'ASIN', 'Style']

[Data Types]
Order ID             object
Date                 object
Status               object
Fulfilment           object
Sales Channel        object
Category             object
Size                 object
Qty                 float64
Amount              float64
ship-city            object
ship-state           object
ship-postal-code      int64
B2B                    bool
promotion-ids        object
ASIN                 object
Style                object

[First 3 rows]
              Order ID                 Date                        Status Fulfilment Sales Channel Category Size  Qty  Amount ship-city  ship-state  ship-postal-code    B2B promotion-ids        ASIN    Style
0  4